# Exploring the GRC Insights database

Run the setup cell first. Every query cell uses `q("...")`, which runs SQL against `grc.duckdb` and returns a pandas DataFrame, so you can keep going in pandas (`.plot()`, `.groupby()`, etc.).

The database is opened **read-only**, so nothing you do here can damage it. Rebuild it any time with `python scripts/build_db.py`.

**Before committing this notebook:** clear the outputs (Kernel → Restart & Clear Output, or `jupyter nbconvert --clear-output --inplace notebooks/explore.ipynb`) so GRC's real numbers don't end up in git.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

SAMPLE = False   # True = explore the fake sample data instead

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
db = root / ("sample_data/sample.duckdb" if SAMPLE else "data/grc.duckdb")
con = duckdb.connect(str(db), read_only=True)

def q(sql):
    """Run SQL, get a DataFrame back."""
    return con.sql(sql).df()

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
print(f"Connected to {db}")

## 1. What's in here?

In [ ]:
q("SHOW TABLES")

In [ ]:
# Column names and types for one table
q("DESCRIBE mc_campaigns")

In [ ]:
# One-shot profile: min, max, distinct values, % null for every column
q("SUMMARIZE email_posts")

In [ ]:
# Look at actual rows
q("SELECT * FROM mc_campaigns ORDER BY sent_at DESC LIMIT 5")

## 2. Saved queries + a quick chart
Any `.sql` file in `sql/` can be run from here, then plotted with pandas.

In [ ]:
df = q((root / "sql" / "channel_open_rates.sql").read_text())
df["quarter"] = pd.to_datetime(df["quarter"]).dt.to_period("Q").astype(str)   # '2025Q1' labels for the x-axis
df.plot(x="quarter", y=["mailchimp_open_rate", "substack_open_rate"], kind="bar",
        figsize=(9, 4), title="Open rate by quarter")
df

## 3. Your turn

Try each one before asking for the answer. They're different kinds of problems on purpose.

**Q1 (grain).** For each table, what does *one row* represent? Write it as a sentence, e.g. "one row = one Substack post." Hint: `mc_list_growth` is *not* one row per contact. Why does knowing the grain matter before you `SUM` or `COUNT`?

In [ ]:
# Q1


**Q2 (skepticism).** Find the 5 Mailchimp campaigns with the highest `open_rate`. Would you tell your CMO the top one is GRC's best-performing email? What filter would you add, and what threshold would you pick?

In [ ]:
# Q2


**Q3 (aggregation).** Which Mailchimp audience has the highest overall open rate? Which has the most unsubscribes per 1,000 delivered? (Remember Problem E: how should rates be combined?)

In [ ]:
# Q3


**Q4 (window functions).** "Applications Open for GMT" was sent six times in spring 2026. Number the sends 1–6 in time order and show open rate and unsubscribes per 1,000 for each. Did re-sending cost subscribers?

In [ ]:
# Q4


**Q5 (time series).** Using `mc_list_growth`, in which months does the main list (`global-research-and-consulting`) gain the most contacts? What does that pattern say about when GRC should launch things?

In [ ]:
# Q5
